In [177]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import yfinance as yf

In [178]:
stocks_name = ['BTC-USD', 'ETH-USD', 'SPY', 'GLD', '^VIX']

In [179]:
stocks_data = yf.download(stocks_name, start='2018-01-01', end=pd.Timestamp.now().date(), interval='1wk') #interval='5d')

[                       0%                       ]

[*********************100%***********************]  5 of 5 completed


In [180]:
df_stocks = pd.DataFrame(stocks_data).dropna()

In [181]:
df_stocks_close = df_stocks['Close'].dropna().drop(columns=['^VIX']).copy()

Normalized BTC/USD, SPY and GLD

In [182]:
normalized_df = (df_stocks_close / df_stocks_close.iloc[0])

In [183]:
fig = make_subplots(
    rows=1,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[1.5]
)

fig.add_trace(go.Scatter(
    x=normalized_df.index,
    y=normalized_df['BTC-USD']
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=normalized_df.index,
    y=normalized_df['SPY']
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=normalized_df.index,
    y=normalized_df['GLD']
), row=1, col=1)

fig.update_layout(
    title='Normalized Prices (BTC, SPY, GLD)',
    xaxis_title='Date',
    yaxis_title='Normalized Price',
    height=600,
    showlegend=True,
    template='plotly_dark'
)

fig.show()

Copy the DataFrame to a new variable

In [184]:
df_stocks_close_criptos = df_stocks['Close'].copy()

Thechnical Indicators

In [185]:
###
# Calculate SMAs for BTC-USD
df_stocks_close_criptos['BTC-USD_EMA-50'] = df_stocks_close_criptos['BTC-USD'].ewm(span=50).mean()
df_stocks_close_criptos['BTC-USD_EMA-100'] = df_stocks_close_criptos['BTC-USD'].ewm(span=100).mean()
df_stocks_close_criptos['BTC-USD_EMA-200'] = df_stocks_close_criptos['BTC-USD'].ewm(span=200).mean()
###
# Calculate SMAs for ETH-USD
df_stocks_close_criptos['ETH-USD_EMA-50'] = df_stocks_close_criptos['ETH-USD'].ewm(span=50).mean()
df_stocks_close_criptos['ETH-USD_EMA-100'] = df_stocks_close_criptos['ETH-USD'].ewm(span=100).mean()
df_stocks_close_criptos['ETH-USD_EMA-200'] = df_stocks_close_criptos['ETH-USD'].ewm(span=200).mean()
###

MACD

In [186]:
###
#MACD
def calculate_macd(df, fast=12, slow=26, signal=9):
    exp1 = df_stocks['Close']['BTC-USD'].ewm(span=fast).mean()
    exp2 = df_stocks['Close']['BTC-USD'].ewm(span=slow).mean()
    macd = exp1 - exp2
    signal = macd.ewm(span=signal).mean()
    histogram = macd - signal
    return macd, signal, histogram


macd, signal, histogram = calculate_macd(df_stocks)

In [187]:
###
#MACD SPY
def calculate_macd_spy(df, fast=12, slow=26, signal=9):
    exp1 = df_stocks['Close']['SPY'].ewm(span=fast).mean()
    exp2 = df_stocks['Close']['SPY'].ewm(span=slow).mean()
    macd = exp1 - exp2
    signal = macd.ewm(span=signal).mean()
    histogram = macd - signal
    return macd, signal, histogram


macd_spy, signal_spy, histogram_spy = calculate_macd_spy(df_stocks)

BB %B - Bollinger Bands Percentage Bandwidth

In [188]:
window = 20
std_dev = 2

price = df_stocks["Close"]["BTC-USD"]
middle = price.rolling(window).mean()
std = price.rolling(window).std()
upper = middle + std_dev * std
lower = middle - std_dev * std

bb_percent_b = (price - lower) / (upper - lower)

In [189]:
window_spy = 20
std_dev_spy = 2

price_spy = df_stocks["Close"]["SPY"]
middle_spy = price_spy.rolling(window_spy).mean()
std_spy = price_spy.rolling(window_spy).std()
upper_spy = middle_spy + std_dev_spy * std_spy
lower_spy = middle_spy - std_dev_spy * std_spy

bb_percent_spy = (price_spy - lower_spy) / (upper_spy - lower_spy)

Stochastic + RSI

In [190]:
# RSI
def calculate_rsi(prices, window=14):
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Stochastic Oscillator
def calculate_stochastic(high, low, close, k_window=14, d_window=3):
    lowest_low = low.rolling(window=k_window).min()
    highest_high = high.rolling(window=k_window).max()
    k_percent = 100 * ((close - lowest_low) / (highest_high - lowest_low))
    d_percent = k_percent.rolling(window=d_window).mean()
    return k_percent, d_percent

# Calcular para BTC-USD
rsi_btc = calculate_rsi(df_stocks['Close']['BTC-USD'])
stoch_k_btc, stoch_d_btc = calculate_stochastic(
    df_stocks['High']['BTC-USD'],
    df_stocks['Low']['BTC-USD'],
    df_stocks['Close']['BTC-USD']
)

In [191]:
# RSI
def calculate_rsi(prices, window=14):
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Stochastic Oscillator
def calculate_stochastic(high, low, close, k_window=14, d_window=3):
    lowest_low = low.rolling(window=k_window).min()
    highest_high = high.rolling(window=k_window).max()
    k_percent = 100 * ((close - lowest_low) / (highest_high - lowest_low))
    d_percent = k_percent.rolling(window=d_window).mean()
    return k_percent, d_percent

# Calcular para SPY
rsi_spy = calculate_rsi(df_stocks['Close']['SPY'])
stoch_k_spy, stoch_d_spy = calculate_stochastic(
    df_stocks['High']['SPY'],
    df_stocks['Low']['SPY'],
    df_stocks['Close']['SPY']
)

Returns of the investments in multiplication

In [192]:
stocks_returns_week_df = (df_stocks_close.pct_change().dropna())#Day variation
acumulated_returns = (1 + stocks_returns_week_df).cumprod()

Log Return

In [193]:
stocks_returns_log_df = np.log(df_stocks_close/ df_stocks_close.shift(1)).dropna()

Volatility Weeks Sharpe & Sortino

In [194]:
trading_weeks_sharpe = 60 #30
trading_weeks_sortino = 90
volatility_weeks_sharpe = stocks_returns_log_df.rolling(window=trading_weeks_sharpe).std()*np.sqrt(trading_weeks_sharpe)
volatility_weeks_sortino = stocks_returns_log_df.rolling(window=trading_weeks_sortino).std()*np.sqrt(trading_weeks_sortino)

Sharpe Ratio Weeks(52)

In [195]:
rf = 0.01/52
sharpe_ratio = (stocks_returns_log_df.rolling(window=trading_weeks_sharpe).mean() - rf)*trading_weeks_sharpe/volatility_weeks_sharpe

Sortino Ratio

In [196]:
sortino_vol = stocks_returns_log_df[stocks_returns_log_df<0].rolling(window=trading_weeks_sortino, center=True, min_periods=10).std()*np.sqrt(trading_weeks_sortino)
sortino_ratio = (stocks_returns_log_df.rolling(window=trading_weeks_sortino).mean() - rf)*trading_weeks_sortino/sortino_vol

Data Conditioning of BTC and SPY

In [197]:
# %% [markdown]
# ## Data Conditioning BTC

nv_shp_btc_top = 2.2
nv_srt_btc_top = 3.5
nv_shp_btc_bottom = -1.5
nv_srt_btc_bottom = -2.0
mask_btc_sortino_top = sortino_ratio['BTC-USD'] >= 3.7
mask_btc_sharpe_top = sharpe_ratio['BTC-USD'] > nv_shp_btc_top
mask_btc_sharpe_sortino_top = ((sortino_ratio['BTC-USD'] >= nv_srt_btc_top) & mask_btc_sharpe_top)
mask_btc_sortino_reversion_down = (sharpe_ratio['BTC-USD'] < 1.2) & (sortino_ratio['BTC-USD'] >= nv_srt_btc_top) | ((sharpe_ratio['BTC-USD'] > nv_shp_btc_top) & (sortino_ratio['BTC-USD'] >= nv_srt_btc_top))
mask_btc_sharpe_bottom = sharpe_ratio['BTC-USD'] < nv_shp_btc_bottom
mask_btc_sortino_bottom = (sortino_ratio['BTC-USD'] < nv_srt_btc_bottom) & mask_btc_sharpe_bottom
mask_btc_sharpe_sortino_bottom = ((sortino_ratio['BTC-USD'] < nv_srt_btc_bottom) & mask_btc_sharpe_bottom)
mask_btc_sortino_reversion_up = (sharpe_ratio['BTC-USD'] > 0) & (sortino_ratio['BTC-USD'] <= -1.5)

# %% [markdown]
# ## Data Conditioning SPY
# %%

nv_shp_spy_top = 2.5
nv_srt_spy_top = 3.2
nv_shp_spy_bottom = 0
nv_srt_spy_bottom = 0
mask_spy_sortino_top = sortino_ratio['SPY'] >= nv_srt_spy_top
mask_spy_sharpe_top = sharpe_ratio['SPY'] > nv_shp_spy_top
mask_spy_sharpe_sortino_top = ((sortino_ratio['SPY'] >= nv_srt_spy_top) & mask_spy_sharpe_top)
mask_spy_sortino_reversion_down = (sharpe_ratio['SPY'] < nv_shp_spy_bottom) & (sortino_ratio['SPY'] >= 3.0)
mask_spy_sharpe_bottom = sharpe_ratio['SPY'] < -0.6
mask_spy_sortino_bottom = (sortino_ratio['SPY'] < nv_srt_spy_bottom)
mask_spy_sharpe_sortino_bottom = ((sortino_ratio['SPY'] < 0) & mask_spy_sharpe_bottom)
mask_spy_sortino_reversion_up = (sharpe_ratio['SPY'] > 1.0) & (sortino_ratio['SPY'] <= 0)


In [198]:
def plot_top_now_btc():
    points_top_now = []
    if (sharpe_ratio['BTC-USD'].iloc[-1] > 1.75) & (sortino_ratio['BTC-USD'].iloc[-1] > 3.0) & (bb_percent_b.iloc[-1] > 0.9) & (rsi_btc.iloc[-1] > 80):
        print(f"BTC-USD atende aos critérios de venda por múltiplos indicadores em {df_stocks.index[-1].date()}")
        points_top_now.append(len(df_stocks['Close'])-1)
    if(sharpe_ratio['BTC-USD'].iloc[-1] > 3.0):
        print(f"BTC-USD atende aos critérios de venda por Sharpe em {df_stocks.index[-1].date()}")
        points_top_now.append(len(df_stocks['Close'])-1)
    else:
        print(f'BTC-USD não atende aos critérios de venda em {df_stocks.index[-1].date()}')
    return points_top_now

def plot_bottom_now_btc():
    points_bottom_now = []
    if (sharpe_ratio['BTC-USD'].iloc[-1] <= -1.4) & (sortino_ratio['BTC-USD'].iloc[-1] <= -1.5) & (bb_percent_b.iloc[-1] <= 0.25) & (rsi_btc.iloc[-1] <= 26):
        print(f"BTC-USD atende aos critérios de compra em {df_stocks.index[-1].date()}")
        points_bottom_now.append(len(df_stocks['Close'])-1)
    else:
        print(f'BTC-USD não atende aos critérios de compra em {df_stocks.index[-1].date()}')
    return points_bottom_now

def plot_top_btc():

    points_top = []
    for date in range(len(df_stocks['Close'])-1):

        if (sharpe_ratio['BTC-USD'].iloc[date] > 1.75) & (sortino_ratio['BTC-USD'].iloc[date] > 3.0) & (bb_percent_b.iloc[date] > 0.9) & (rsi_btc.iloc[date] > 80):
            points_top.append(date+1)
        if (sharpe_ratio['BTC-USD'].iloc[date] > 3.0):
            points_top.append(date+1)
    return points_top

def plot_bottom_btc():

    points_bottom = []
    for date in range(len(df_stocks['Close'])-1):
        if (sharpe_ratio['BTC-USD'].iloc[date] <= -1.4) & (sortino_ratio['BTC-USD'].iloc[date] <= -1.5) & (bb_percent_b.iloc[date] <= 0.25) & (rsi_btc.iloc[date] <= 28):
            points_bottom.append(date-1)
    return points_bottom

top_points_btc_now = plot_top_now_btc()
bottom_points_btc_now = plot_bottom_now_btc()

top_points_btc = plot_top_btc()
bottom_points_btc = plot_bottom_btc()

print("Top points BTC:", top_points_btc)
print("Bottom points BTC:", bottom_points_btc)

BTC-USD não atende aos critérios de venda em 2026-03-30
BTC-USD não atende aos critérios de compra em 2026-03-30
Top points BTC: [174, 322, 323, 324, 325, 326, 362, 363]
Bottom points BTC: [253, 257]


In [199]:
def plot_top_now_spy():
    points_top_now = []
    if (sharpe_ratio['SPY'].iloc[-1] > 1.7) & (sortino_ratio['SPY'].iloc[-1] > 3.2) & (bb_percent_spy.iloc[-1] > 0.8) & (rsi_spy.iloc[-1] > 70):
        points_top_now.append(len(df_stocks['Close'])-1)
        print(f"SPY atende aos critérios de venda em {df_stocks.index[-1].date()}")
    if (sharpe_ratio['SPY'].iloc[-1] > 1.9) & (sortino_ratio['SPY'].iloc[-1] > 4):
        points_top_now.append(len(df_stocks['Close'])-1)
        print(f"SPY atende aos critérios de venda em {df_stocks.index[-1].date()}")
    if (sharpe_ratio['SPY'].iloc[-1] > 2.5):
        points_top_now.append(len(df_stocks['Close'])-1)
        print(f"SPY atende aos critérios de venda em {df_stocks.index[-1].date()}")
    else:
        print(f'SPY não atende aos critérios de venda em {df_stocks.index[-1].date()}')
    return points_top_now

def plot_bottom_now_spy():
    points_bottom_now = []
    if (sharpe_ratio['SPY'].iloc[-1] < 0) & (sortino_ratio['SPY'].iloc[-1] < 0) & (bb_percent_spy.iloc[-1] < 0.3) & (rsi_spy.iloc[-1] < 45):
        points_bottom_now.append(len(df_stocks['Close'])-1)
        print(f"SPY atende aos critérios de compra em {df_stocks.index[-1].date()}")
    else:
        print(f'SPY não atende aos critérios de compra em {df_stocks.index[-1].date()}')
    return points_bottom_now

def plot_top_spy():

    points_top = []
    for date in range(len(df_stocks['Close'])-1):
        if (sharpe_ratio['SPY'].iloc[date] > 1.9) & (sortino_ratio['SPY'].iloc[date] > 3.2) & (bb_percent_spy.iloc[date] > 0.8) & (rsi_spy.iloc[date] > 75):
            points_top.append(date-1)
        if (sharpe_ratio['SPY'].iloc[date] > 1.9) & (sortino_ratio['SPY'].iloc[date] > 4):
            points_top.append(date-1)
        if (sharpe_ratio['SPY'].iloc[date] > 2.5):
            points_top.append(date-1)
    return points_top

def plot_bottom_spy():

    points_bottom = []
    for date in range(len(df_stocks['Close'])-1):
        if (sharpe_ratio['SPY'].iloc[date] < 0) & (sortino_ratio['SPY'].iloc[date] < 0) & (bb_percent_spy.iloc[date] < 0.3) & (rsi_spy.iloc[date] < 45):
            points_bottom.append(date)
    return points_bottom


top_points_spy_now = plot_top_now_spy()
bottom_points_spy_now = plot_bottom_now_spy()

top_points_spy = plot_top_spy()
bottom_points_spy = plot_bottom_spy()

print(f'Top points: {top_points_spy}')
print(f'Bottom points: {bottom_points_spy}')

SPY não atende aos critérios de venda em 2026-03-30
SPY não atende aos critérios de compra em 2026-03-30
Top points: [108, 109, 173, 175, 203, 205, 205, 322, 360, 361, 362]
Bottom points: [114, 115, 116, 247, 248, 270]


Graphic

In [202]:
# %% [markdown]
# ## Data Visualization
# %%
fig = make_subplots(
    rows=10,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.005,
    row_heights=[4.5, 3.5, 3.5, 3.5, 3.5, 3.5, 3.5, 3.5, 3.5, 3.5]
)

# %% [markdown]
# ## Data Plotting Prices
# %%
fig.add_trace(go.Candlestick(
    x=df_stocks.index,
    open=df_stocks['Open']['BTC-USD'],
    high=df_stocks['High']['BTC-USD'],
    low=df_stocks['Low']['BTC-USD'],
    close=df_stocks['Close']['BTC-USD'],
    name='BTC-USD',

), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_stocks.index[top_points_btc],
    y=df_stocks['Close']['BTC-USD'].iloc[top_points_btc],
    mode='markers+text',
    marker=dict(size=10,
                color='lightcoral',
                symbol='circle-dot',),
    name=f'Top Points ({len(top_points_btc)} points)'),
    row=1, col=1
)

if len(top_points_btc_now) != 0:
    fig.add_trace(go.Scatter(
        x=df_stocks.index[top_points_btc_now],
        y=df_stocks['Close']['BTC-USD'].iloc[top_points_btc_now],
        mode='markers+text',
        marker=dict(size=12,
                    color='lightcoral',
                    symbol='x',),
        name=f'Top Points (now) ({len(top_points_btc_now)} points)'),
        row=1, col=1
    )

fig.add_trace(go.Scatter(
    x=df_stocks.index[top_points_spy],
    y=df_stocks['Close']['BTC-USD'].iloc[top_points_spy],
    mode='markers+text',
    marker=dict(size=10,
                color='red',
                symbol='star',
                opacity=0.7),
    name=f'Top Points ({len(top_points_spy)} points)'),
    row=1, col=1
)

if len(top_points_btc_now) != 0:
    fig.add_trace(go.Scatter(
        x=df_stocks.index[top_points_spy_now],
        y=df_stocks['Close']['BTC-USD'].iloc[top_points_spy_now],
        mode='markers+text',
        marker=dict(size=12,
                    color='red',
                    symbol='x',),
        name=f'Top Points (now) ({len(top_points_spy_now)} points)'),
        row=1, col=1
    )

fig.add_trace(go.Scatter(
    x=df_stocks.index[bottom_points_btc],
    y=df_stocks['Close']['BTC-USD'].iloc[bottom_points_btc],
    mode='markers+text',
    marker=dict(size=10,
                color='lightgreen',
                symbol='circle-dot'),
    name=f'Bottom Points ({len(bottom_points_btc)} points)'),
    row=1, col=1
)

if len(top_points_btc_now) != 0:
    fig.add_trace(go.Scatter(
        x=df_stocks.index[bottom_points_btc_now],
        y=df_stocks['Close']['BTC-USD'].iloc[bottom_points_btc_now],
        mode='markers+text',
        marker=dict(size=12,
                    color='lightcoral',
                    symbol='x',),
        name=f'Bottom Points (now) ({len(bottom_points_btc_now)} points)'),
        row=1, col=1
    )

fig.add_trace(go.Scatter(
    x=df_stocks.index[bottom_points_spy],
    y=df_stocks['Close']['BTC-USD'].iloc[bottom_points_spy],
    mode='markers+text',
    marker=dict(size=10,
                color='green',
                symbol='star',
                opacity=0.7),
    name=f'Bottom Points ({len(bottom_points_spy)} points)'),
    row=1, col=1
)

if len(top_points_btc_now) != 0:
    fig.add_trace(go.Scatter(
        x=df_stocks.index[bottom_points_spy_now],
        y=df_stocks['Close']['BTC-USD'].iloc[bottom_points_spy_now],
        mode='markers+text',
        marker=dict(size=12,
                    color='red',
                    symbol='x',),
        name=f'Bottom Points (now) ({len(bottom_points_spy_now)} points)'),
        row=1, col=1
    )

###
# Plot BTC SMAs
fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_EMA-50'],
    name='EMA-50',
    line=dict(color='lightgreen')
), row=1, col=1)


fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_EMA-100'],
    name='EMA-100',
    line=dict(color='lightblue')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_stocks_close_criptos.index,
    y=df_stocks_close_criptos['BTC-USD_EMA-200'],
    name='EMA-200',
    line=dict(color='white')
), row=1, col=1)

# %% [markdown]
# ## Data Plotting MACD BTC
# %%
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=macd,
    name='MACD',
    marker_color='lightblue',
    opacity=0.7
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=signal,
    name='Signal',
    marker_color='lightcoral',
    opacity=0.7
), row=2, col=1)
fig.add_trace(go.Bar(
    x=df_stocks.index,
    y=histogram,
    name='Histogram',
    marker_color=histogram,
), row=2, col=1)

# %% [markdown]
# ## Data Plotting MACD SPY
# %%

fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=macd_spy,
    name='MACD SPY',
    marker_color='lightblue',
    opacity=0.7
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=signal_spy,
    name='Signal SPY',
    marker_color='lightcoral',
    opacity=0.7
), row=3, col=1)
fig.add_trace(go.Bar(
    x=df_stocks.index,
    y=histogram_spy,
    name='Histogram',
    marker_color=histogram_spy,
), row=3, col=1)

# %% [markdown]
# ## Data Plotting Sharpe and Sortino Ratios BTC
# %%

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index,
    y=sharpe_ratio['BTC-USD'],
    line=dict(color='white'),
    name='Sharpe Ratio BTC',
), row=4, col=1)

fig.add_trace(go.Bar(
    x=sortino_ratio.index,
    y=sortino_ratio['BTC-USD'],
    marker_color=sortino_ratio['BTC-USD'],
    name='Sortino Ratio BTC',
), row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sharpe_sortino_top],
    y=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top])*4,
                color=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_top],
                colorscale='YlOrRd',
                symbol='diamond'),
    name='Sortino BTC > 3.5 & Sharpe > 2.0'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sharpe_sortino_bottom],
    y=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom])*6,
                color=sortino_ratio['BTC-USD'][mask_btc_sharpe_sortino_bottom],
                colorscale='BuGn',
                symbol='diamond'),
    name='Sort < -2.0 & Shar < -2.0'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_btc_sharpe_top],
    y=sharpe_ratio['BTC-USD'][mask_btc_sharpe_top],
    mode='markers',
    marker=dict(size=sharpe_ratio['BTC-USD'][mask_btc_sharpe_top]*3,
                color='lightcoral',
                symbol='circle'),
    name='Sharpe BTC > 2'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_top],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_top],
    mode='markers',
    marker=dict(size=sortino_ratio['BTC-USD'][mask_btc_sortino_top]*2,
                color=sortino_ratio['BTC-USD'][mask_btc_sortino_top],
                colorscale='PuRd',
                symbol='circle'),
    name='Sortino BTC > 3.5'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_btc_sharpe_bottom],
    y=sharpe_ratio['BTC-USD'][mask_btc_sharpe_bottom],
    mode='markers',
    marker=dict(size=abs(sharpe_ratio['BTC-USD'][mask_btc_sharpe_bottom])*3,
                color='lightgreen',
                symbol='circle'),
    name='Sharpe BTC < -2'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_reversion_down],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down])*2,
                color=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_down],
                colorscale='YlOrRd',
                symbol='circle'),
    name=f'Sort > 3.5 & Shar < 0'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_bottom],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_bottom])*3,
                color='green',
                symbol='circle'),
    name=f'Sortino BTC < -2.0'),
    row=4, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_btc_sortino_reversion_up],
    y=sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_up],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['BTC-USD'][mask_btc_sortino_reversion_up])*5,
                color='lightblue',
                symbol='star'),
    name='Sort < -1.5 & Shar > 0'),
    row=4, col=1
)

# %% [markdown]
# ## Data Plotting Sharpe and Sortino Ratios SPY
# %%

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index,
    y=sharpe_ratio['SPY'],
    line=dict(color='white'),
    name='Sharpe Ratio SPY',
), row=5, col=1)

fig.add_trace(go.Bar(
    x=sortino_ratio.index,
    y=sortino_ratio['SPY'],
    marker_color=sortino_ratio['SPY'],
    name='Sortino Ratio SPY',
), row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sharpe_sortino_top],
    y=sortino_ratio['SPY'][mask_spy_sharpe_sortino_top],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sharpe_sortino_top])*3,
                color=sortino_ratio['SPY'][mask_spy_sharpe_sortino_top],
                colorscale='YlOrRd',
                symbol='diamond'),
    name='Sortino SPY > 3.2 & Sharpe > 2.5'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sharpe_sortino_bottom],
    y=sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom])*35,
                color=sortino_ratio['SPY'][mask_spy_sharpe_sortino_bottom],
                colorscale='BuGn',
                symbol='diamond'),
    name='Sort < 0 & Shar < -0.2'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_spy_sharpe_top],
    y=sharpe_ratio['SPY'][mask_spy_sharpe_top],
    mode='markers',
    marker=dict(size=sharpe_ratio['SPY'][mask_spy_sharpe_top]*3,
                color='lightcoral',
                symbol='circle'),
    name='Sharpe SPY > 2'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_top],
    y=sortino_ratio['SPY'][mask_spy_sortino_top],
    mode='markers',
    marker=dict(size=sortino_ratio['SPY'][mask_spy_sortino_top]*2,
                color=sortino_ratio['SPY'][mask_spy_sortino_top],
                colorscale='YlOrRd',
                symbol='circle'),
    name='Sortino SPY > 3.2'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sharpe_ratio.index[mask_spy_sharpe_bottom],
    y=sharpe_ratio['SPY'][mask_spy_sharpe_bottom],
    mode='markers',
    marker=dict(size=abs(sharpe_ratio['SPY'][mask_spy_sharpe_bottom])*8,
                color='lightgreen',
                symbol='circle'),
    name='Sharpe SPY < -0.2'),
    row=5, col=1
)


fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_reversion_down],
    y=sortino_ratio['SPY'][mask_spy_sortino_reversion_down],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sortino_reversion_down])*2,
                color='red',
                symbol='circle'),
    name=f'Sort > 3.5'),
    row=5, col=1
)

fig.add_trace(go.Scatter(
    x=sortino_ratio.index[mask_spy_sortino_bottom],
    y=sortino_ratio['SPY'][mask_spy_sortino_bottom],
    mode='markers',
    marker=dict(size=abs(sortino_ratio['SPY'][mask_spy_sortino_bottom])*12,
                color='green',
                symbol='circle'),
    name=f'Sortino SPY < 0'),
    row=5, col=1
)

# %% [markdown]
# ## Data Plotting Bollinger Bands %B BTC-USD
# %%
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=bb_percent_b,
    name='BB %B BTC',
    line=dict(color='white', width=1),
    opacity=0.7,
), row=6, col=1)

fig.update_layout(
    yaxis6=dict(
        autorange=False,
        range=[-1, 1.5],
        title="BB %B BTC"
    )
)

fig.add_hline(y=0, line_dash="dot", line_color="lightgreen", row=6, col=1)
fig.add_hline(y=0.50, line_dash="dot", line_color="gray", row=6, col=1)
fig.add_hline(y=1, line_dash="dot", line_color="lightcoral", row=6, col=1)

# %% [markdown]
# ## Data Plotting Bollinger Bands %B SPY
# %%
fig.add_trace(go.Scatter(
    x=df_stocks.index,
    y=bb_percent_spy,
    name='BB %B SPY',
    line=dict(color='white', width=1),
    opacity=0.7,
), row=7, col=1)

fig.update_layout(
    yaxis7=dict(
        autorange=False,
        range=[-1, 1.5],
        title="BB %B SPY"
    )
)

fig.add_hline(y=0, line_dash="dot", line_color="lightgreen", row=7, col=1)
fig.add_hline(y=0.50, line_dash="dot", line_color="gray", row=7, col=1)
fig.add_hline(y=1, line_dash="dot", line_color="lightcoral", row=7, col=1)

# %% [markdown]
# ## Data Plotting RSI and Stochastic BTC
# %%
fig.add_trace(go.Scatter(
            x=df_stocks.index,
            y=rsi_btc,
            name='RSI BTC',
            line=dict(color='white', width=1)
), row=8, col=1)

fig.add_trace(go.Scatter(
            x=df_stocks.index,
            y=stoch_d_btc,
            name='%D BTC',
            line=dict(color='red', width=1)
), row=8, col=1)

fig.add_hline(y=75, line_dash="dot", line_color="lightcoral", row=8, col=1)
fig.add_hline(y=50, line_dash="dot", line_color="gray", row=8, col=1)
fig.add_hline(y=25, line_dash="dot", line_color="lightgreen", row=8, col=1)


# %% [markdown]
# ## Data Plotting RSI and Stochastic SPY
# %%
fig.add_trace(go.Scatter(
            x=df_stocks.index,
            y=rsi_spy,
            name='RSI SPY',
            line=dict(color='white', width=1)
), row=9, col=1)

fig.add_trace(go.Scatter(
            x=df_stocks.index,
            y=stoch_d_spy,
            name='%D SPY',
            line=dict(color='red', width=1)
), row=9, col=1)

fig.add_hline(y=75, line_dash="dot", line_color="lightcoral", row=9, col=1)
fig.add_hline(y=50, line_dash="dot", line_color="gray", row=9, col=1)
fig.add_hline(y=25, line_dash="dot", line_color="lightgreen", row=9, col=1)


# %% [markdown]
# ## Data Plotting VIX
# %%

fig.add_trace(go.Bar(
    x=df_stocks.index,
    y=df_stocks['Close']['^VIX'],
    name='VIX',
    marker_color=df_stocks['Close']['^VIX']
), row=10, col=1)

# %% [markdown]
# ## Data Plotting End
# %%

fig.update_layout(
    title= f"Market Analysis - {df_stocks.index[0].date().strftime('%Y-%m-%d')} to {df_stocks.index[-1].date().strftime('%Y-%m-%d')}",
    title_x=0.5,
    showlegend=False,
    height=2000,
    width=1150,
    yaxis_title='Price',
    yaxis_type='log',
    yaxis2_title='MACD BTC',
    yaxis3_title='MACD SPY',
    yaxis4_title='SHP&STO BTC',
    yaxis4_type='linear',
    yaxis5_title='SHP&STO SPY',
    yaxis5_type='linear',
    yaxis6_title='BB %B BTC',
    yaxis6_type='linear',
    yaxis7_title='BB %B SPY',
    yaxis7_type='linear',
    yaxis8_title='RSI & STO BTC',
    yaxis8_type='linear',
    yaxis9_title='RSI & STO SPY',
    yaxis9_type='linear',
    yaxis10_title='VIX',
    template='plotly_dark',
    xaxis_rangeslider_visible=False,


)

fig.show()